Regresión lineal - Sujay G Kaushik et al.

In [18]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
import pandas as pd
from src.data.loader import load_raw_data

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score


1. Cargar datos


In [11]:
# La ruta relativa es desde la raíz del proyecto
df = load_raw_data("../data/raw/db.csv")
df.head()

,Description,Date,Amount,Area,Type
0,Streaming plataforma digital,2026-02-28,10.00,Leisure,Expenses
1,Nómina mensual trabajo,2026-02-27,629.58,Salary,Income
2,Alquiler bici turismo,2026-02-25,51.00,Vacations,Expenses
3,Salida bar copas,2026-02-22,2.80,Leisure,Expenses
4,Compra diaria alimentación,2026-02-21,24.80,Food,Expenses


2. Procesar datos

In [13]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['month'] = df['Date'].dt.month
df['year'] = df['Date'].dt.year
df['day'] = df['Date'].dt.day
df.head()

,Description,Date,Amount,Area,Type,month,year,day
0,Streaming plataforma digital,2026-02-28,10.00,Leisure,Expenses,2,2026,28
1,Nómina mensual trabajo,2026-02-27,629.58,Salary,Income,2,2026,27
2,Alquiler bici turismo,2026-02-25,51.00,Vacations,Expenses,2,2026,25
3,Salida bar copas,2026-02-22,2.80,Leisure,Expenses,2,2026,22
4,Compra diaria alimentación,2026-02-21,24.80,Food,Expenses,2,2026,21


In [14]:
#Filtrar solo gastos
df_expenses = df[df['Type'] == 'Expenses'].copy()

In [16]:
df_expenses.head()

,Description,Date,Amount,Area,Type,month,year,day
0,Streaming plataforma digital,2026-02-28,10.0,Leisure,Expenses,2,2026,28
2,Alquiler bici turismo,2026-02-25,51.0,Vacations,Expenses,2,2026,25
3,Salida bar copas,2026-02-22,2.8,Leisure,Expenses,2,2026,22
4,Compra diaria alimentación,2026-02-21,24.8,Food,Expenses,2,2026,21
5,Pedido comida online,2026-02-21,21.6,Food,Expenses,2,2026,21


In [ ]:
# Gasto acumulado de los primeros 10 días (X)
gasto_10d = df_expenses[df_expenses['day'] <= 10].groupby(['year', 'month'])['Amount'].sum().reset_index()

# Gasto total de todo el mes (y)
gasto_total = df_expenses.groupby(['year', 'month'])['Amount'].sum().reset_index()

# Pares (X, y)
df_ml = pd.merge(gasto_10d, gasto_total, on=['year', 'month'], suffixes=('_10d', '_total'))

# Definir matrices
X = df_ml[['Amount_10d']] 
y = df_ml['Amount_total']

print(f"Tenemos {len(df_ml)} meses de datos para entrenar.")
df_ml.head()

Tenemos 193 meses de datos para entrenar.


,year,month,Amount_10d,Amount_total
0,2010,2,16.62,77.85
1,2010,3,86.17,288.42
2,2010,4,15.51,133.17
3,2010,5,93.51,168.51
4,2010,6,57.14,266.54


3. Modelo

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = LinearRegression()
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

4. Resultados

In [ ]:
print(f"Precisión (R2 Score): {r2_score(y_test, y_pred):.2f}")
print(f"Error Medio Absoluto (MAE): {mean_absolute_error(y_test, y_pred):.2f}€")

Precisión (R2 Score): 0.33
Error Medio Absoluto (MAE): 81.52€
